In [42]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import numpy as np
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.fit_params.weibull_fit_params import WeibullFitParams

plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [43]:
power_law_db = pl.read_parquet(r"MLE_random_sample_fit_data\PowerLaw")
weibull_db = pl.read_parquet(r"MLE_random_sample_fit_data\Weibull") # .filter(pl.col("k_err").is_not_nan())
log_normal_db = pl.read_parquet(r"MLE_random_sample_fit_data\LogNormal") # .filter(pl.col("mu_err").is_not_nan())

In [44]:
power_law_db

seed,aic,bic,numb_alphas,s_max_fitting_alpha,s_min_fitting_alpha,q,g_mu,g_std,q_err,g_mu_err,g_std_err,LAD_min,J_min,min_alpha_to_consider
i32,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64
39188,2.6018e6,2.6019e6,192734,34166.764882,22.776133,1.045848,0.424517,0.078808,NaN,NaN,NaN,0,0.7,100.0
87298,2.6057e6,2.6057e6,192734,34166.764882,23.497827,1.073605,0.41723,0.07811,NaN,NaN,NaN,0,0.7,100.0
24401,2.6044e6,2.6044e6,192734,34166.764882,22.776133,1.040998,0.415775,0.080857,NaN,NaN,NaN,0,0.7,100.0
12872,2.6041e6,2.6042e6,192734,34166.764882,22.776133,1.014706,0.425428,0.078133,NaN,NaN,NaN,0,0.7,100.0
43138,2.5958e6,2.5959e6,192734,34166.764882,22.776133,1.039501,0.424164,0.079211,NaN,NaN,NaN,0,0.7,100.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
56320,8343.922225,8367.657249,386,34166.764882,22.776133,1.773448,0.282997,0.045751,NaN,NaN,NaN,0,0.7,20000.0
75733,8344.671281,8368.406306,386,34166.764882,23.497827,1.636319,0.293129,0.047974,NaN,NaN,NaN,0,0.7,20000.0
84705,8345.14717,8368.882194,386,34166.764882,22.776133,1.9293,0.269514,0.044265,NaN,NaN,NaN,0,0.7,20000.0


In [45]:
columns_to_select = ["aic", "min_alpha_to_consider"]

aic_db = pl.concat([
    power_law_db.select(columns_to_select).with_columns(pl.lit("PL").alias("fit_type")),
    weibull_db.select(columns_to_select).with_columns(pl.lit("W").alias("fit_type")),
    log_normal_db.select(columns_to_select).with_columns(pl.lit("LN").alias("fit_type"))
]).filter(pl.col("min_alpha_to_consider") == 20000)

aic_db

aic,min_alpha_to_consider,fit_type
f64,f64,str
8345.18332,20000.0,"""PL"""
8342.057548,20000.0,"""PL"""
8348.296266,20000.0,"""PL"""
8345.66443,20000.0,"""PL"""
8350.722825,20000.0,"""PL"""
…,…,…
8343.345637,20000.0,"""W"""
8337.251908,20000.0,"""W"""
8343.460624,20000.0,"""W"""


In [46]:
aic_db = aic_db.join(
        aic_db.group_by("min_alpha_to_consider").agg(pl.col("aic").min().alias("min_aic_for_min_alpha_to_consider")),
        on = "min_alpha_to_consider",
    ).with_columns(
        (pl.col("aic") - pl.col("min_aic_for_min_alpha_to_consider")).alias("delta_aic")
    )

aic_db

aic,min_alpha_to_consider,fit_type,min_aic_for_min_alpha_to_consider,delta_aic
f64,f64,str,f64,f64
8345.18332,20000.0,"""PL""",8335.04626,10.13706
8342.057548,20000.0,"""PL""",8335.04626,7.011288
8348.296266,20000.0,"""PL""",8335.04626,13.250006
8345.66443,20000.0,"""PL""",8335.04626,10.61817
8350.722825,20000.0,"""PL""",8335.04626,15.676565
…,…,…,…,…
8343.345637,20000.0,"""W""",8335.04626,8.299377
8337.251908,20000.0,"""W""",8335.04626,2.205648
8343.460624,20000.0,"""W""",8335.04626,8.414365


In [47]:
aic_db.group_by("fit_type").agg(
    pl.col("delta_aic").mean().alias("rel_mean_aic"),
    pl.col("delta_aic").std().alias("std_aic"),
    pl.col("delta_aic").min().alias("rel_min_aic"),
    pl.col("delta_aic").max().alias("rel_max_aic"),
    pl.len()
)

fit_type,rel_mean_aic,std_aic,rel_min_aic,rel_max_aic,len
str,f64,f64,f64,f64,u32
"""PL""",11.607725,3.97288,4.589144,21.180794,60
"""W""",7.535095,4.001484,0.0,19.479685,180


In [48]:
import polars as pl

types = aic_db["fit_type"].unique().sort().to_list()

samples = {
    t: aic_db.filter(pl.col("fit_type") == t)["delta_aic"].to_numpy()
    for t in types
}

for t in types:
    others = [o for o in types if o != t]

    a = samples[t]
    b = samples[others[0]]
    c = samples[others[1]]

    p = (
        (a[:, None, None] < b[None, :, None]) &
        (a[:, None, None] < c[None, None, :])
    ).mean()

    print(f"P(AIC {t} < ({others[0]}, {others[1]})) = {100*p:.3f}%")

IndexError: list index out of range